In [ ]:
from pathlib import Path
import os, sys
WORKSPACE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/branch_sql_MVP/settings.json').is_file())
sys.path.insert(0, str(WORKSPACE)) if str(WORKSPACE) not in sys.path else None
MVP_ROOT = WORKSPACE / 'src/branch_sql_MVP'
os.chdir(MVP_ROOT)


# Đánh giá split nội bộ v1

Kết quả cũ là exploratory/legacy, không phải kết quả của split này. Dev dùng để chọn thông số, internal test chỉ chạy sau khi khóa cấu hình. Đây là đánh giá câu hỏi mới trên database đã biết, không phải official BIRD test.

Business: mỗi bảng là một chunk; mỗi Q là một chunk riêng. Gold `-- Query:`: query child → nguyên parent Query/Evidence/SQL; dev/test gold không được index.

Notebook chỉ đọc artifact, không gọi model.

In [1]:
import json
from pathlib import Path
from collections import Counter
import pandas as pd
from IPython.display import display
from src.branch_sql_MVP.data.catalog import load_benchmark_cases
from src.branch_sql_MVP.eval.dataset_split import question_key

ROOT = MVP_ROOT
split = json.loads((ROOT / 'eval/dataset_split.json').read_text(encoding='utf-8'))
cases = load_benchmark_cases(ROOT / 'data/dev', 'dev')
lookup = {c.stable_id: c for c in cases}
dev = [lookup[k] for k in split['dev_ids']]
test = [lookup[k] for k in split['test_ids']]
assert not set(split['dev_ids']) & set(split['test_ids'])
assert not {question_key(c) for c in dev} & {question_key(c) for c in test}
assert not set(split['previously_run_ids']) & set(split['test_ids'])
display(pd.DataFrame([{'role': role, 'db': db, 'difficulty': difficulty, 'cases': n} for role, rows in [('dev', dev), ('internal_test', test)] for (db, difficulty), n in Counter((c.db_id, c.difficulty) for c in rows).items()]))
print('Dev:', len(dev), 'Internal test:', len(test), 'Legacy overlapping questions:', split['legacy_question_overlap'])

,role,db,difficulty,cases
0,dev,thrombosis_prediction,simple,41
1,dev,california_schools,simple,43
2,dev,california_schools,moderate,24
3,dev,california_schools,challenging,4
4,dev,card_games,simple,100
...,...,...,...,...
61,internal_test,thrombosis_prediction,simple,9
62,internal_test,thrombosis_prediction,challenging,6
63,internal_test,toxicology,simple,15
64,internal_test,toxicology,moderate,8


Dev: 1227 Internal test: 307 Legacy overlapping questions: 12


In [2]:
path = ROOT / '.runtime/luna_internal_holdout_v1/luna_component_tuning.json'
if path.exists():
    bundle = json.loads(path.read_text(encoding='utf-8'))
    display(bundle['protocol'])
    display(bundle['parameter_space'])
    display(bundle['selected'])
    records = []
    for stage, rows in bundle['dev_tuning'].items():
        for row in rows:
            records.append({'stage': stage, 'candidate': row.get('name', row.get('value')), 'run_id': row['run_id'], 'parameters': row['manifest']['parameters'], **row['metrics'], **row.get('proxy_metrics', {})})
    display(pd.DataFrame(records))
    display(pd.DataFrame([{'label': r['label'], 'run_id': r['run_id'], 'parameters': r['manifest']['parameters'], **r['metrics']} for r in bundle['test_scenarios']]))
else:
    print('Chưa chạy model trên split mới; chưa có accuracy mới. Không dùng lại điểm legacy.')

{'order': 'coordinate search on dev only, freeze once, evaluate all test scenarios',
 'selection_metric_order': ['execution_accuracy:max',
  'run_error_rate:min',
  'invalid_sql_rate:min',
  'input_tokens_all_calls:min',
  'output_tokens_all_calls:min'],
 'dev_total_cases': 1534,
 'dev_tuning_cases': 99,
 'dev_sampling': 'up to 3 per database/difficulty; 0 means all',
 'dev_pool_cases': 1227,
 'split_manifest': 'src/eval/dataset_split.json',
 'test_cases': 307,
 'test_scope': 'all internal holdout cases; source IDs retain dev prefix; legacy 62 excluded',
 'provider': 'openai',
 'model': 'gpt-5.6-luna',
 'reasoning_effort': 'selected on dev',
 'workflow_revision': 'internal-holdout-v1',
 'temperature': None,
 'evaluation': 'sqlite_result_equivalence_v1',
 'r_ves': None,
 'token_limit': 9000000,
 'token_ledger': '.runtime\\luna_internal_holdout_v1\\token_ledger.jsonl'}

{'base_context': {'mode': 'hybrid',
  'candidate_k': 16,
  'docs_top_k': 5,
  'semantic_weight': 0.5,
  'keyword_weight': 0.5,
  'rrf_k': 40,
  'rerank_top_k': 5,
  'rerank_enabled': False,
  'rerank_max_length': 128,
  'rerank_batch_size': 8,
  'table_k': 5,
  'column_k': 12,
  'schema_min_score': 0.12,
  'value_k': 8,
  'value_fuzzy_threshold': 0.84,
  'token_budget': 5000,
  'event_top_k': 8,
  'event_hops': 1,
  'event_node_budget': 24,
  'event_token_budget': 1800,
  'selector_max_items': 24,
  'example_k': 0},
 'reasoning_grid': ['low', 'medium', 'high'],
 'context_stages': [{'name': 'retrieval_fusion',
   'scenario': 'B4',
   'proxy': 'linked',
   'candidates': [{'name': 'dense_only',
     'mode': 'semantic',
     'semantic_weight': 1.0,
     'keyword_weight': 0.0},
    {'name': 'keyword_only',
     'mode': 'keyword',
     'semantic_weight': 0.0,
     'keyword_weight': 1.0},
    {'name': 'hybrid_keyword_0_7',
     'mode': 'hybrid',
     'semantic_weight': 0.3,
     'keyword_weig

{'reasoning_effort': {'value': 'high',
  'label': 'component-reasoning-high',
  'run_id': 'dev-P1-01a85ad62f64',
  'scenario': 'P1',
  'manifest': {'run_id': 'dev-P1-01a85ad62f64',
   'scenario': 'P1',
   'split': 'dev',
   'model_provider': 'openai',
   'model': 'gpt-5.6-luna',
   'prompt_version': 'bird-sql-v1',
   'dataset_fingerprint': 'f177dc09b400b66e48df250cb3643a5e5039b13fbdc0889c7fba1dab6d92c117',
   'index_fingerprint': '3600417e2ef9d29421869a16b46c1417a37664edeacda124c7aef18e176271d5',
   'dependency_lock_sha256': '40483697b00ca0eb2b03d2fbfcfefbf22914d1da5e0de0444efe7dc34d41aa6b',
   'seed': 42,
   'hardware': {'platform': 'Windows-11-10.0.26200-SP0',
    'python': '3.12.13',
    'torch': '2.11.0+cu128',
    'cuda_build': '12.8',
    'cuda_available': True,
    'device': 'NVIDIA GeForce RTX 4050 Laptop GPU'},
   'parameters': {'context': {},
    'execution': {'timeout_seconds': 5.0,
     'max_rows': 500,
     'eval_timeout_seconds': 10.0},
    'route': {},
    'max_repairs':

,stage,candidate,run_id,parameters,cases,completed_cases,execution_accuracy,evaluable_cases,evaluation_error_cases,evaluation_coverage,...,mean_candidates,r_ves,r_ves_reason,input_tokens_all_calls,output_tokens_all_calls,schema_table_recall,schema_complete_rate,mean_evidence_items,mean_context_tokens,elapsed_seconds
0,reasoning_effort,low,dev-P1-a746ab321dc7,"{'context': {}, 'execution': {'timeout_seconds...",99,99,0.469388,98,1,0.989899,...,1.0,None,Chưa tích hợp official BIRD R-VES evaluator; k...,845964,20819,NaN,NaN,NaN,NaN,NaN
1,reasoning_effort,medium,dev-P1-6a109ad56119,"{'context': {}, 'execution': {'timeout_seconds...",99,99,0.469388,98,1,0.989899,...,1.0,None,Chưa tích hợp official BIRD R-VES evaluator; k...,845964,27542,NaN,NaN,NaN,NaN,NaN
2,reasoning_effort,high,dev-P1-01a85ad62f64,"{'context': {}, 'execution': {'timeout_seconds...",99,99,0.479592,98,1,0.989899,...,1.0,None,Chưa tích hợp official BIRD R-VES evaluator; k...,845964,48187,NaN,NaN,NaN,NaN,NaN
3,retrieval_fusion,dense_only,dev-B4-5ea25b995213,"{'context': {'mode': 'semantic', 'candidate_k'...",99,99,0.448980,98,1,0.989899,...,1.0,None,Chưa tích hợp official BIRD R-VES evaluator; k...,356877,63775,0.980471,0.949495,37.717172,2996.626263,78.115072
4,retrieval_fusion,keyword_only,dev-B4-4d959d666fd0,"{'context': {'mode': 'keyword', 'candidate_k':...",99,99,0.438776,98,1,0.989899,...,1.0,None,Chưa tích hợp official BIRD R-VES evaluator; k...,409437,54386,0.982997,0.959596,37.484848,3411.636364,22.576095
5,retrieval_fusion,hybrid_keyword_0_7,dev-B4-07242d7aa280,"{'context': {'mode': 'hybrid', 'candidate_k': ...",99,99,0.428571,98,1,0.989899,...,1.0,None,Chưa tích hợp official BIRD R-VES evaluator; k...,380330,57297,0.980471,0.949495,37.606061,3179.060606,39.097218
6,retrieval_fusion,hybrid_balanced,dev-B4-46f2181cf99f,"{'context': {'mode': 'hybrid', 'candidate_k': ...",99,99,0.448980,98,1,0.989899,...,1.0,None,Chưa tích hợp official BIRD R-VES evaluator; k...,376025,49550,0.980471,0.949495,37.626263,3144.909091,45.918320
7,retrieval_fusion,hybrid_dense_0_7,dev-B4-94950de9bc38,"{'context': {'mode': 'hybrid', 'candidate_k': ...",99,99,0.479592,98,1,0.989899,...,1.0,None,Chưa tích hợp official BIRD R-VES evaluator; k...,371394,58215,0.980471,0.949495,37.666667,3110.121212,55.657103
8,reranking,incumbent_before_reranking,dev-B4-94950de9bc38,"{'context': {'mode': 'hybrid', 'candidate_k': ...",99,99,0.479592,98,1,0.989899,...,1.0,None,Chưa tích hợp official BIRD R-VES evaluator; k...,371394,58215,0.980471,0.949495,37.666667,3110.121212,56.543821
9,reranking,rerank_off,dev-B4-94950de9bc38,"{'context': {'mode': 'hybrid', 'candidate_k': ...",99,99,0.479592,98,1,0.989899,...,1.0,None,Chưa tích hợp official BIRD R-VES evaluator; k...,371394,58215,0.980471,0.949495,37.666667,3110.121212,58.183065


""


In [3]:
runtime = ROOT / '.runtime/luna_internal_holdout_v1'
ledger_path = runtime / 'token_ledger.jsonl'
ledger = [json.loads(line) for line in ledger_path.read_text(encoding='utf-8').splitlines() if line.strip()] if ledger_path.exists() else []
confirmed_input = sum(r.get('usage', {}).get('input_tokens', 0) for r in ledger)
confirmed_output = sum(r.get('usage', {}).get('output_tokens', 0) for r in ledger)
accounted = sum(r['delta'] for r in ledger)
display(pd.DataFrame([{'input_confirmed': confirmed_input, 'output_confirmed': confirmed_output, 'accounted_including_reservations': accounted, 'limit': 9000000}]))
progress = []
for run_path in sorted((runtime / 'runs').glob('*/cases.jsonl')):
    entries = [json.loads(line) for line in run_path.read_text(encoding='utf-8').splitlines() if line.strip()]
    rows = list({r['stable_id']: r for r in entries}.values())
    progress.append({'run_id': run_path.parent.name, 'saved_cases': len(rows), 'run_errors': sum(bool(r.get('run_error')) for r in rows), 'has_summary': (run_path.parent / 'summary.json').exists()})
display(pd.DataFrame(progress))
print('Partial runs are progress only, never treated as completed experiment scores.')

,input_confirmed,output_confirmed,accounted_including_reservations,limit
0,7854498,887078,8975825,9000000


,run_id,saved_cases,run_errors,has_summary
0,dev-B4-00f3ab365b1d,99,0,True
1,dev-B4-07242d7aa280,99,0,True
2,dev-B4-1f8bf838c583,99,0,True
3,dev-B4-2a3869af2e37,99,0,True
4,dev-B4-323c5202c132,99,0,True
5,dev-B4-46f2181cf99f,99,0,True
6,dev-B4-4d959d666fd0,99,0,True
7,dev-B4-5ea25b995213,99,0,True
8,dev-B4-67da8f425c9a,99,0,True
9,dev-B4-6831423ff977,99,0,True


Partial runs are progress only, never treated as completed experiment scores.


## Tạm dừng theo trần 9 triệu token — 2026-09-07
17 run hoàn chỉnh trên 99 dev; value_linking/values_off đang dở 15/99. Chưa chạy internal test, chưa có cấu hình hoàn chỉnh. Best-so-far sau schema: 49/98 = 50%; gold codebase_community:701 timeout nên không chấm, coverage 98/99.

Usage API xác nhận 7.854.498 input + 887.078 output = 8.741.576 token. Ledger bảo thủ 8.975.825 bao gồm 234.249 token dự phòng chưa đối soát từ lỗi/gián đoạn. Không suy ra billing hoặc quota toàn tài khoản từ ledger này.

Đợt chính concurrency=6, lượt resume cuối concurrency=1. Manifest hiện thể hiện giới hạn concurrency yêu cầu cuối do runner ghi lại khi resume; không dùng các số latency này để kết luận về concurrency. Kết quả accuracy và SQL đã cache vẫn được giữ.

Tiếp tục khi có quota mới: giữ split/ledger/cache; hoàn tất các stage dev còn lại rồi mới khóa cấu hình và chạy 307 internal test. Không sử dụng điểm của run đang dở để chọn winner.

## Ước tính hoàn tất — 2026-09-08 (chưa cấp thêm ngân sách)

Căn cứ: token thực tế các stage cũ 33 dev, nhân 99/33; trừ cache/case đã chạy và dự phòng khác biệt cấu hình mới. Dev còn lại thô 16.393.557 token; kế hoạch 15–17 triệu. Test cũ quy đổi 307/62 là 13.084.637 token, nhưng khác phân bố DB, P1 context và selector nên dự trù 15–19 triệu cho 8×307 case-run. Tổng cần thêm khoảng 30–36 triệu token; đây là ước tính, không phải usage đã phát sinh.

Dự trù 5–8 giờ với concurrency=6 và cache resume, phụ thuộc API, GPU và SQLite. Chi phí niêm yết khoảng 9–13 USD với tỷ lệ input/output tương tự các run đã có; chưa trừ quota miễn phí hoặc cache discount. Giá tham khảo: https://developers.openai.com/api/docs/models/gpt-5.6-luna ($0.20/M input, $1.20/M output).

Đề xuất trần mới hôm nay 36.000.000 token, tương ứng trần ledger tích lũy 44.975.825. Chưa áp dụng vì người dùng đã yêu cầu giới hạn 9 triệu; cần xác nhận phần ngân sách tăng thêm trước khi gọi API. Đã thêm reuse proxy từ bundle và giữ summary/manifest của run hoàn chỉnh khi resume.